# 📸 GFPGAN Premium Enhancer

**Mejora tus fotos con inteligencia artificial directamente desde la nube.**

Este notebook te permite restaurar y mejorar fotos de baja resolución usando GFPGAN, aprovechando la GPU gratuita de Google Colab.

> ⚠️ **Importante:** Antes de empezar, activa la GPU gratuita:
> 1. Ve a **Entorno de ejecución** (Runtime) en el menú superior
> 2. Haz clic en **Cambiar tipo de entorno de ejecución** (Change runtime type)
> 3. Selecciona **T4 GPU** en "Acelerador de hardware"
> 4. Haz clic en **Guardar**

## Paso 1: Instalar dependencias 📦
Ejecuta esta celda para clonar el repositorio e instalar todo lo necesario. Toma ~2 minutos.

In [ ]:
import os, subprocess, site, sys

# Verificar si ya estamos dentro del repo
if not os.path.exists('inference_gfpgan.py'):
    if os.path.exists('GFPGAN'):
        os.chdir('GFPGAN')
    else:
        subprocess.run(['git', 'clone', 'https://github.com/Bitwolf89/GFPGAN.git'])
        os.chdir('GFPGAN')

print(f'Directorio actual: {os.getcwd()}')

# Instalar dependencias
!pip install basicsr facexlib realesrgan gfpgan -q
!pip install -r requirements.txt -q
!python setup.py develop -q

# ============================================
# PARCHE: Corregir incompatibilidad de basicsr
# con versiones nuevas de torchvision
# ============================================
import importlib
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    degradations_file = os.path.join(sp, 'basicsr', 'data', 'degradations.py')
    if os.path.exists(degradations_file):
        with open(degradations_file, 'r') as f:
            content = f.read()
        if 'functional_tensor' in content:
            content = content.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            with open(degradations_file, 'w') as f:
                f.write(content)
            print(f'🔧 Parche aplicado en: {degradations_file}')
        break

# Tambien parchear functional_tensor en el directorio local
local_deg = os.path.join('gfpgan', '..', 'basicsr', 'data', 'degradations.py')
# Buscar recursivamente
for root, dirs, fnames in os.walk(sys.prefix):
    if 'degradations.py' in fnames and 'basicsr' in root:
        fpath = os.path.join(root, 'degradations.py')
        with open(fpath, 'r') as f:
            c = f.read()
        if 'functional_tensor' in c:
            c = c.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            with open(fpath, 'w') as f:
                f.write(c)
            print(f'🔧 Parche adicional aplicado en: {fpath}')

# Verificar GPU
import torch
if torch.cuda.is_available():
    print(f'\n✅ Instalación completada! GPU detectada: {torch.cuda.get_device_name(0)}')
else:
    print('\n⚠️ Instalación completada pero NO se detectó GPU.')

# Test de importación
try:
    from gfpgan import GFPGANer
    print('✅ GFPGANer import OK')
except Exception as e:
    print(f'❌ Error importando GFPGANer: {e}')

## Paso 2: Descargar modelos pre-entrenados 🧠
Descargamos el modelo V1.4 (el más reciente y nítido).

In [ ]:
!mkdir -p experiments/pretrained_models

# Descargar modelo GFPGAN v1.4
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P experiments/pretrained_models

# (Opcional) Descomenta para descargar otros modelos
# !wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P experiments/pretrained_models
# !wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/RestoreFormer.pth -P experiments/pretrained_models

print('✅ Modelo descargado!')
!ls -lh experiments/pretrained_models/

## Paso 3: Subir tus fotos 🖼️
Sube las fotos que quieras mejorar. Se guardarán en la carpeta `inputs/upload`.

In [ ]:
import os
from google.colab import files

# Crear carpeta para las fotos subidas
upload_folder = 'inputs/upload'
os.makedirs(upload_folder, exist_ok=True)

# Subir fotos (se abre un selector de archivos)
print('📂 Selecciona las fotos que quieras mejorar...')
uploaded = files.upload()

# Mover a la carpeta de inputs
for filename in uploaded.keys():
    dest = os.path.join(upload_folder, filename)
    if os.path.exists(filename):
        os.rename(filename, dest)
    print(f'  ✅ {filename} subida correctamente')

print(f'\n🎉 {len(uploaded)} foto(s) lista(s) para mejorar!')
!ls -lh inputs/upload/

## Paso 4: ¡Mejorar las fotos! 🚀
Ejecuta la restauración con Python directamente (más confiable que shell).

In [ ]:
import cv2
import glob
import numpy as np
import os
import torch

# === CONFIGURACIÓN ===
VERSION = '1.4'        # Opciones: '1.2', '1.3', '1.4', 'RestoreFormer'
ESCALA = 2             # 2 = doble resolución, 4 = cuádruple
MEJORAR_FONDO = True   # True para mejorar fondo con Real-ESRGAN
PESO = 0.5             # 0.0 = más original, 1.0 = más restaurado
# =====================

# --- Configurar Background Upsampler ---
bg_upsampler = None
if MEJORAR_FONDO:
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        bg_upsampler = RealESRGANer(
            scale=2,
            model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
            model=model,
            tile=400, tile_pad=10, pre_pad=0,
            half=True)
        print('✅ Real-ESRGAN (fondo) cargado')
    except Exception as e:
        print(f'⚠️ No se pudo cargar Real-ESRGAN: {e}')
        print('   Continuando sin mejora de fondo...')

# --- Configurar GFPGAN ---
from gfpgan import GFPGANer

models = {
    '1.2': ('clean', 2, 'https://github.com/TencentARC/GFPGAN/releases/download/v0.2.0/GFPGANCleanv1-NoCE-C2.pth'),
    '1.3': ('clean', 2, 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'),
    '1.4': ('clean', 2, 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'),
    'RestoreFormer': ('RestoreFormer', 2, 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/RestoreFormer.pth'),
}

arch, channel_multiplier, model_url = models[VERSION]

# Buscar modelo local primero
model_path = f'experiments/pretrained_models/GFPGANv{VERSION}.pth'
if not os.path.isfile(model_path):
    model_path = model_url  # descarga automática

restorer = GFPGANer(
    model_path=model_path,
    upscale=ESCALA,
    arch=arch,
    channel_multiplier=channel_multiplier,
    bg_upsampler=bg_upsampler)
print(f'✅ GFPGAN v{VERSION} cargado')

# --- Procesar fotos ---
input_folder = 'inputs/upload'
img_list = sorted(glob.glob(os.path.join(input_folder, '*')))
print(f'\n📷 Encontradas {len(img_list)} foto(s) para procesar...\n')

os.makedirs('results/restored_imgs', exist_ok=True)
os.makedirs('results/cropped_faces', exist_ok=True)
os.makedirs('results/restored_faces', exist_ok=True)
os.makedirs('results/cmp', exist_ok=True)

for img_path in img_list:
    img_name = os.path.basename(img_path)
    print(f'  Procesando {img_name} ...', end=' ')
    basename, ext = os.path.splitext(img_name)
    input_img = cv2.imread(img_path, cv2.IMREAD_COLOR)

    if input_img is None:
        print(f'⚠️ No se pudo leer (formato no soportado)')
        continue

    try:
        cropped_faces, restored_faces, restored_img = restorer.enhance(
            input_img, has_aligned=False, only_center_face=False,
            paste_back=True, weight=PESO)

        # Guardar imagen restaurada completa
        if restored_img is not None:
            save_path = os.path.join('results/restored_imgs', f'{basename}{ext}')
            cv2.imwrite(save_path, restored_img)
            print(f'✅ Guardada ({restored_img.shape[1]}x{restored_img.shape[0]})')
        else:
            print('⚠️ Sin resultado')

        # Guardar caras individuales
        for idx, (cropped, restored) in enumerate(zip(cropped_faces, restored_faces)):
            cv2.imwrite(os.path.join('results/cropped_faces', f'{basename}_{idx:02d}.png'), cropped)
            cv2.imwrite(os.path.join('results/restored_faces', f'{basename}_{idx:02d}.png'), restored)
            cmp_img = np.concatenate((cropped, restored), axis=1)
            cv2.imwrite(os.path.join('results/cmp', f'{basename}_{idx:02d}.png'), cmp_img)

    except Exception as e:
        print(f'❌ Error: {e}')

count = len(os.listdir('results/restored_imgs'))
print(f'\n🎉 ¡{count} foto(s) mejorada(s) exitosamente!')

## Paso 5: Ver resultados 👀
Compara las fotos originales con las mejoradas lado a lado.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob
import os

restored_imgs = sorted(glob.glob('results/restored_imgs/*'))
input_imgs = sorted(glob.glob('inputs/upload/*'))

if not restored_imgs:
    print('❌ No se encontraron fotos mejoradas. Ejecuta el Paso 4 primero.')
else:
    for inp_path, res_path in zip(input_imgs, restored_imgs):
        img_input = cv2.imread(inp_path)
        img_input = cv2.cvtColor(img_input, cv2.COLOR_BGR2RGB)
        img_result = cv2.imread(res_path)
        img_result = cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(18, 9))
        axes[0].imshow(img_input)
        axes[0].set_title('🖼️ Original', fontsize=16)
        axes[0].axis('off')
        axes[1].imshow(img_result)
        axes[1].set_title('✨ Mejorada', fontsize=16)
        axes[1].axis('off')
        fig.suptitle(os.path.basename(inp_path), fontsize=14, y=0.02)
        plt.tight_layout()
        plt.show()

    print(f'🌟 Se procesaron {len(restored_imgs)} foto(s)')

## Paso 6: Descargar resultados 📥
Descarga todas las fotos mejoradas como un archivo ZIP.

In [ ]:
import shutil
import os
from google.colab import files

if not os.path.exists('results/restored_imgs') or len(os.listdir('results/restored_imgs')) == 0:
    print('❌ No hay fotos mejoradas para descargar.')
    print('   Asegúrate de ejecutar los Pasos 3 y 4 primero.')
else:
    shutil.make_archive('fotos_mejoradas', 'zip', 'results')
    zip_size = os.path.getsize('fotos_mejoradas.zip') / (1024 * 1024)
    num_files = len(os.listdir('results/restored_imgs'))
    print(f'📦 ZIP creado: {num_files} foto(s), {zip_size:.1f} MB')
    files.download('fotos_mejoradas.zip')
    print('🎉 ¡Descarga iniciada! Revisa tu carpeta de descargas.')

---
## 📚 Tips

| Modelo | Fortaleza | Mejor para |
|--------|-----------|------------|
| **v1.4** | Más detalles, mejor identidad | Uso general ⭐ |
| **v1.3** | Resultados más naturales | Fotos muy dañadas |
| **v1.2** | Más nítido, con maquillaje | Retratos |
| **RestoreFormer** | Otra arquitectura | Experimentar |

> 💡 **PESO**: Ajusta el valor de `PESO` en el Paso 4. Más cerca de 0 = más fiel al original. Más cerca de 1 = más restaurado.

---
*Fork de [TencentARC/GFPGAN](https://github.com/TencentARC/GFPGAN) • Mejorado por Bitwolf89*